# Tool Calling with Local Models

Tool calling = LLM using a tool

What does it mean to use a tool?

It means the model produces an output that indicates something along the lines of: "use_tool_X()" and
code written by humans processes that request.

In [1]:
from ollama import chat
from pydantic import BaseModel

class Country(BaseModel):
  name: str
  capital: str
  languages: list[str]

response = chat(
  model='gemma4',
  messages=[{'role': 'user', 'content': 'Tell me about Canada.'}],
  format=Country.model_json_schema(),
)

country = Country.model_validate_json(response.message.content)
print(country)

name='Canada' capital='Ottawa' languages=['English', 'French', 'Indigenous Languages...']


In [2]:
response.message.content

'{"name": "Canada", "capital": "Ottawa", "languages": ["English", "French", "Indigenous Languages..."]}'

In [4]:
from pydantic import BaseModel

class PhoneData(BaseModel):
  name: str
  brand: str
  price: float
  additional_notes: str | None = None

with open('phone_data.txt', 'r') as f:
  raw_data = f.read()

response = chat(
  model='gemma4',
  messages=[{'role': 'user', 'content': f'Extract the phone data info: {raw_data}'}],
  format=PhoneData.model_json_schema(),
)

response

ChatResponse(model='gemma4', created_at='2026-10-01T18:02:17.181402Z', done=True, done_reason='stop', total_duration=21158891250, load_duration=30113708, prompt_eval_count=5591, prompt_eval_duration=4773828000, eval_count=1015, eval_duration=16274511000, message=Message(role='assistant', content='{"name": "iPhone Duo", "brand": "Apple", "price": 2399, "additional_notes": "The device is a pioneering folding smartphone (dobrável). All data is derived from the provided Portuguese text. Specific release dates mentioned are available for reservation starting Oct 16, 13h, and available on Oct 23."}', thinking='Here\'s a thinking process to extract the phone data:\n\n1.  **Understand the Goal:** The request is to "Extract the phone data info" from a large block of text (which is a product landing page for the "iPhone Duo").\n\n2.  **Analyze the Source Text:** The text is a detailed, promotional description of the iPhone Duo, containing specifications, features, pricing, and model comparisons.

In [8]:
response.message.content

'{"name": "iPhone Duo", "brand": "Apple", "price": 2399, "additional_notes": "The device is a pioneering folding smartphone (dobrável). All data is derived from the provided Portuguese text. Specific release dates mentioned are available for reservation starting Oct 16, 13h, and available on Oct 23."}'

How bout tools?

In [16]:
class ReadFileTool(BaseModel):
  file_path: str


def read_file(file_path: str) -> str:
  with open(file_path, 'r') as f:
    return f.read()

response = chat(
  model='gemma4',
  messages=[{'role': 'user', 'content': 'Read the file at ./translate_local.py and summarize its contents.'}],
  format=ReadFileTool.model_json_schema(),
)  

response.message.content

'{\n    "file_path": "./translate_local.py"\n}'

In [17]:
input_read_file_tool = ReadFileTool.model_validate_json(response.message.content)

In [18]:
input_read_file_tool.file_path

'./translate_local.py'

In [19]:
file_path = input_read_file_tool.file_path

read_file(file_path)

'# /// script\n# requires-python = ">=3.10"\n# dependencies = [\n#     "ollama>=0.6.3",\n# ]\n# ///\nfrom ollama import chat\nfrom ollama import ChatResponse\nimport argparse\n\nMODEL = \'translategemma:4b\'\n\ndef translate_text(text: str, target_language: str) -> str:\n    """\n    Translates the given text into the target language using the Gemma4 model.\n\n    Args:\n        text (str): The text to be translated.\n        target_language (str): The language to translate the text into.\n\n    Returns:\n        str: The translated text.\n    """\n    response: ChatResponse = chat(\n        model=MODEL,\n        messages=[\n            {\n                \'role\': \'user\',\n                \'content\': f\'\'\'Translate the following text into {target_language}: "{text}". \n                Your output should be ONLY the translation verbatim and nothing else.\'\'\',\n            },\n        ],\n    )\n    return response.message.content\n\ndef parse_text_input(input_str: str) -> str:\n

In [20]:
def read_file(file_path: str) -> str:
    """Reads the contents of a file 
    at the given file path."""
    with open(file_path, 'r') as f:
        return f.read()

output = chat(
  model='gemma4',
  messages=[{'role': 'user', 'content': f'Read the file at {file_path} and summarize its contents.'}],
  tools=[read_file]
)
output

ChatResponse(model='gemma4', created_at='2026-10-01T18:08:37.915558Z', done=True, done_reason='stop', total_duration=1776781833, load_duration=30320500, prompt_eval_count=85, prompt_eval_duration=172737000, eval_count=112, eval_duration=1567727000, message=Message(role='assistant', content='', thinking='The user wants me to read a file located at `./translate_local.py` and then provide a summary of its contents.\nI need to use the `read_file` tool for this task.\n\n1.  **Tool selection**: `read_file`.\n2.  **Argument**: `file_path` = `./translate_local.py`.\n3.  **Execution**: Call the tool.', images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='read_file', arguments={'file_path': './translate_local.py'}))]), logprobs=None)

In [26]:
output.message.tool_calls[0].function.name

'read_file'

In [ ]:
file_path = output.message.tool_calls[0].function.arguments['file_path']

'./translate_local.py'

In [29]:
read_file(file_path)

'# /// script\n# requires-python = ">=3.10"\n# dependencies = [\n#     "ollama>=0.6.3",\n# ]\n# ///\nfrom ollama import chat\nfrom ollama import ChatResponse\nimport argparse\n\nMODEL = \'translategemma:4b\'\n\ndef translate_text(text: str, target_language: str) -> str:\n    """\n    Translates the given text into the target language using the Gemma4 model.\n\n    Args:\n        text (str): The text to be translated.\n        target_language (str): The language to translate the text into.\n\n    Returns:\n        str: The translated text.\n    """\n    response: ChatResponse = chat(\n        model=MODEL,\n        messages=[\n            {\n                \'role\': \'user\',\n                \'content\': f\'\'\'Translate the following text into {target_language}: "{text}". \n                Your output should be ONLY the translation verbatim and nothing else.\'\'\',\n            },\n        ],\n    )\n    return response.message.content\n\ndef parse_text_input(input_str: str) -> str:\n